# StoryRec — End-to-End Walkthrough

This notebook walks through the two-stage recommendation pipeline interactively.
Run `python pipeline.py` from the project root first so all artifacts exist.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src import config

## 1. The dataset

In [ ]:
stories = pd.read_csv(config.RAW_DIR / 'stories.csv')
users = pd.read_csv(config.RAW_DIR / 'users.csv')
interactions = pd.read_csv(config.RAW_DIR / 'interactions.csv', parse_dates=['timestamp'])
print(f'{len(users):,} users | {len(stories):,} stories | {len(interactions):,} interactions')
stories.sample(5, random_state=1)[['story_id', 'title', 'genre', 'author', 'description']]

## 2. Story embeddings
384-d sentence-transformer vectors. Nearest neighbours in this space are semantically similar stories.

In [ ]:
emb = np.load(config.PROCESSED_DIR / 'story_embeddings.npy')
sid = 0
sims = emb @ emb[sid]
top = np.argsort(-sims)[1:6]
print('Anchor:', stories.iloc[sid].title, '|', stories.iloc[sid].genre)
stories.iloc[top][['title', 'genre']].assign(similarity=sims[top].round(3))

## 3. Two-stage recommendations
FAISS retrieval (Top-100) → LightGBM LambdaRank → Top-10.

In [ ]:
from src.recommender import Recommender
rec = Recommender()
rec.recommend('U00042')[['rank', 'title', 'genre', 'similarity', 'rank_score']]

In [ ]:
rec.user_history('U00042')[['timestamp', 'title', 'genre', 'completion_rate', 'likes']].head(10)

## 4. Offline evaluation results

In [ ]:
import json
json.loads((config.REPORTS_DIR / 'metrics.json').read_text())